<a href="https://colab.research.google.com/github/huynhphatloi/semisub-cxr/blob/main/notebooks/colab_runner.ipynb" target="_parent"><img src="http://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎓 ASAG Grading Model — Fine-tune DeBERTa-v3

Fine-tune `microsoft/deberta-v3-base` for **Automatic Short Answer Grading**.

| | |
|---|---|
| **Input** | `question [SEP] reference_answer [SEP] student_answer` |
| **Output** | 3-way: `correct` / `partially_correct` / `incorrect` |
| **Data** | `data-generate.csv` — 10,000 samples (7,000 train) |
| **Time** | ~15–20 min on T4 GPU |
| **Result** | Model pushed to your Hugging Face Hub |

---
### ⚡ Before you start
1. Make sure **GPU is enabled**: `Runtime → Change runtime type → T4 GPU`
2. Upload `data-generate.csv` to your Google Drive
3. Run cells **top to bottom** — each step builds on the previous

In [ ]:
#@title ⚙️ Step 0 — Install dependencies (run once)
!pip install -q transformers datasets accelerate huggingface_hub scikit-learn pandas sentencepiece protobuf
print("✅ Dependencies installed")

In [ ]:
#@title 🔑 Step 1 — Login to Hugging Face Hub
# Get your free token at: https://huggingface.co/settings/tokens
# Select 'Write' permission
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
#@title 📂 Step 2 — Mount Google Drive and set CSV path
from google.colab import drive
drive.mount('/content/drive')

# ─── CHANGE THIS to where you put data-generate.csv ───────────────
CSV_PATH = '/content/drive/MyDrive/Projects/dataset/question/data-generate.csv'
# ──────────────────────────────────────────────────────────────────

import os
if os.path.exists(CSV_PATH):
    print(f"✅ Found: {CSV_PATH}")
else:
    print(f"❌ Not found: {CSV_PATH}")
    print("   → Upload data-generate.csv to your Google Drive root folder")
    print("   → Or change CSV_PATH above to the correct location")

In [ ]:
#@title 📊 Step 3 — Load and inspect data
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f"Total rows: {len(df):,}")
print(f"\nLabel distribution (label_3way):")
print(df['label_3way'].value_counts())
print(f"\nSplit distribution:")
print(df['split'].value_counts())
print(f"\nSample row:")
print(df[['question', 'student_answer', 'label_3way']].iloc[0])

In [ ]:
#@title ✂️ Step 4 — Prepare train / val / test splits
LABEL_MAP = {'correct': 0, 'partially_correct': 1, 'incorrect': 2}
LABEL_NAMES = ['correct', 'partially_correct', 'incorrect']

# Keep only rows with valid 3-way labels
df_valid = df[df['label_3way'].isin(LABEL_MAP.keys())].copy()
df_valid['label_id'] = df_valid['label_3way'].map(LABEL_MAP)

train_df = df_valid[df_valid['split'] == 'train'].reset_index(drop=True)
val_df   = df_valid[df_valid['split'] == 'valid'].reset_index(drop=True)
test_df  = df_valid[df_valid['split'].str.startswith('test')].reset_index(drop=True)

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
print(f"\nTrain label distribution:")
print(train_df['label_3way'].value_counts())

In [ ]:
#@title 🗂️ Step 5 — Build HuggingFace Dataset
from datasets import Dataset, DatasetDict

def make_input_text(row):
    """Format: question [SEP] reference_answer [SEP] student_answer"""
    q = str(row['question'])        if pd.notna(row['question'])        else ''
    r = str(row['reference_answer']) if pd.notna(row['reference_answer']) else ''
    s = str(row['student_answer'])  if pd.notna(row['student_answer'])  else ''
    return f"{q} [SEP] {r} [SEP] {s}"

def df_to_hf(dataframe):
    texts  = [make_input_text(row) for _, row in dataframe.iterrows()]
    labels = dataframe['label_id'].tolist()
    return Dataset.from_dict({'text': texts, 'label': labels})

dataset = DatasetDict({
    'train':      df_to_hf(train_df),
    'validation': df_to_hf(val_df),
    'test':       df_to_hf(test_df),
})

print(dataset)
print(f"\nExample input (first 200 chars):")
print(dataset['train'][0]['text'][:200])

In [ ]:
#@title 🔤 Step 6 — Tokenize
from transformers import AutoTokenizer

MODEL_NAME = 'microsoft/deberta-v3-base'
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256,
    )

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized.set_format('torch')
print("\n✅ Tokenized dataset:")
print(tokenized)

In [ ]:
#@title 🤖 Step 7 — Load DeBERTa model
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: 'correct', 1: 'partially_correct', 2: 'incorrect'},
    label2id={'correct': 0, 'partially_correct': 1, 'incorrect': 2},
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Parameters: {model.num_parameters():,}")

In [ ]:
#@title ⚙️ Step 8 — Training configuration
# ─── CHANGE THIS to your HF username ──────────────────────────────
HF_USERNAME = 'loihuynh'  # <-- CHANGE THIS
# ──────────────────────────────────────────────────────────────────

REPO_NAME = f'{HF_USERNAME}/asag-grading-deberta-v3'

from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':     accuracy_score(labels, preds),
        'macro_f1':     f1_score(labels, preds, average='macro'),
        'weighted_f1':  f1_score(labels, preds, average='weighted'),
    }

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    push_to_hub=True,
    hub_model_id=REPO_NAME,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    compute_metrics=compute_metrics,
)

print(f"✅ Trainer ready")
print(f"   Epochs:     {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   LR:         {training_args.learning_rate}")
print(f"   Push to:    https://huggingface.co/{REPO_NAME}")

In [ ]:
#@title 🚀 Step 9 — TRAIN  (~15–20 min on T4)
print("🚀 Starting training...")
print("   This will take ~15-20 minutes on a T4 GPU.")
print("   You can watch the loss decrease in the logs below.\n")

trainer.train()

print("\n✅ Training complete!")

🚀 Starting training...
   This will take ~15-20 minutes on a T4 GPU.
   You can watch the loss decrease in the logs below.



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
#@title 📊 Step 10 — Evaluate on test set
print("📊 Evaluating on test set...")
results = trainer.evaluate(tokenized['test'])

print(f"\n{'='*40}")
print(f"  TEST RESULTS")
print(f"{'='*40}")
print(f"  Accuracy:    {results['eval_accuracy']:.4f}")
print(f"  Macro F1:    {results['eval_macro_f1']:.4f}")
print(f"  Weighted F1: {results['eval_weighted_f1']:.4f}")
print(f"{'='*40}")

In [ ]:
#@title 📤 Step 11 — Push model to Hugging Face Hub
print(f"📤 Pushing model to: https://huggingface.co/{REPO_NAME}")

trainer.push_to_hub(
    commit_message="ASAG grading: DeBERTa-v3-base fine-tuned on 7K samples (3-way)"
)
tokenizer.push_to_hub(REPO_NAME)

print(f"\n✅ Done! Your model is live at:")
print(f"   https://huggingface.co/{REPO_NAME}")
print(f"\n📋 Copy this for your .env.local:")
print(f"   HF_MODEL_ID={REPO_NAME}")

In [ ]:
#@title 🧪 Step 12 — Quick inference test
from transformers import pipeline

print("Loading model from Hub for inference test...")
classifier = pipeline(
    'text-classification',
    model=REPO_NAME,
    tokenizer=REPO_NAME,
    device=0,
)

test_cases = [
    {
        'question': 'What is photosynthesis?',
        'reference': 'Plants convert light energy into chemical energy stored in glucose using chlorophyll.',
        'student': 'Plants use sunlight to make food from CO2 and water, producing glucose and oxygen.',
        'expected': 'correct'
    },
    {
        'question': 'What is photosynthesis?',
        'reference': 'Plants convert light energy into chemical energy stored in glucose using chlorophyll.',
        'student': 'Plants use sunlight to make food.',
        'expected': 'partially_correct'
    },
    {
        'question': 'What is photosynthesis?',
        'reference': 'Plants convert light energy into chemical energy stored in glucose using chlorophyll.',
        'student': 'Plants get food from the soil.',
        'expected': 'incorrect'
    },
]

print(f"\n{'='*60}")
print(f"  INFERENCE TEST")
print(f"{'='*60}")
for tc in test_cases:
    text = f"{tc['question']} [SEP] {tc['reference']} [SEP] {tc['student']}"
    result = classifier(text, truncation=True, max_length=256)
    pred   = result[0]['label']
    conf   = result[0]['score']
    match  = '✅' if pred == tc['expected'] else '❌'
    print(f"  {match} Student: '{tc['student'][:45]}...'")
    print(f"     Predicted: {pred} ({conf:.3f}) | Expected: {tc['expected']}")
    print()

---
## ✅ All done!

Your fine-tuned model is now on Hugging Face Hub.

### Next steps — connect to your demo app:

1. Copy your model ID: `YOUR_HF_USERNAME/asag-grading-deberta-v3`
2. In `demos/project1-grading/.env.local`, add:
   ```
   HF_API_TOKEN=hf_xxxxxxxxxxxx
   HF_MODEL_ID=YOUR_HF_USERNAME/asag-grading-deberta-v3
   ```
3. Deploy to Vercel — the grading API will now use your real model

### Free tier limits:
- HF Inference API: ~30K calls/month (more than enough for a thesis demo)
- Model storage: unlimited on HF Hub